# PRISM 26TS08 — Face-Branch Train & Test Pipeline (Colab) · v2

Clean rebuild. Every fix found while debugging v1 on a live Colab box is baked
in, so this should run straight through.

### Protocol
**Train** DFDC (2 parts) → **validate** on a video-disjoint split of it →
**test** on fully held-out **Celeb-DF v2**. A real cross-dataset result, no
contamination.

### Verified on Colab 2026-10-14
| | |
|---|---|
| Python / torch | 3.13 · **2.11.0+cu130** |
| numpy | **2.1.3** (do *not* let anything downgrade this) |
| GPU / disk | Tesla T4 · ~107 GB total |
| Celeb-DF v2 mirror | ✅ 9.29 GB, 6529 mp4 |
| DFDC mirror | ⚠️ **96.5 GB total** — per-part download only |
| FF++ mirror | ❌ PNG frames, and **contains Celeb-DF** → skipped |

### Fixes over v1
1. `facenet-pytorch` installs with **`--no-deps`**. Its metadata pins
   `numpy<2`, `torch<2.3`; honouring those makes pip compile numpy 1.26 from
   source on Python 3.13 and then downgrade torch. Its code has **zero**
   numpy-2 incompatibilities, so the pins are just stale.
2. **kaggle CLI is upgraded first.** 2.0.2 returns `404` on
   `DownloadDataset`; 2.2.4+ works.
3. **DFDC downloads per part.** All 10 parts cannot fit a Colab disk.
4. **FF++ dropped.** Its "DeepfakeBench-ready" mirror ships PNG frames *and*
   Celeb-DF — training on it while testing on Celeb-DF v2 is silent leakage
   that the registry's name-based checks cannot detect.
5. **Crop paths use a stable SHA-1.** v1 used `hash()`, which Python salts per
   process, so resume never matched and every crop re-extracted each session.
6. **Disk guard before each download**, so nothing dies half-written.

> **Scope, stated honestly.** Every dataset in the source spreadsheet is a
> face-deepfake set, so this trains the **face branch**. Your own measurements
> put that branch at **0.64-0.71 AUC cross-dataset** (EXP-001/005) against the
> general AI-video branch's **0.9426** on an unseen generator (EXP-004). This
> is the right tool for Phase 6 (*fix or formally de-scope the face branch*).
> It will **not** improve general AI-video detection — that needs different
> data entirely.

**Reference:** EXP-001 (FF++-trained → Celeb-DF v2) = AUC **0.7076**, recall
0.9471, FPR 0.7247. Yours trains on DFDC, so it's a *new* data point rather
than a replication. In-distribution val will read ~0.99 and means nothing.

## 1 · Runtime

In [ ]:
import subprocess, sys, shutil
gpu = subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True)
print(gpu.stdout.strip() or "NO GPU -- Runtime > Change runtime type > T4 GPU")
t,u,f = shutil.disk_usage("/content")
print(f"disk /content: {f/1e9:.0f} GB free / {t/1e9:.0f} GB total")
print("python", sys.version.split()[0])

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU, then rerun."
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

## 2 · Drive (do not skip)

Colab wipes `/content` on disconnect. This project has already lost a 3-hour
face-crop extraction twice to exactly that. Crops, manifests and checkpoints
live on Drive; only raw video stays local, since it is re-downloadable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

PERSIST   = Path('/content/drive/MyDrive/prism_video')
CROPS     = PERSIST/'crops'; MANIFESTS = PERSIST/'manifests'
CKPTS     = PERSIST/'checkpoints'; REPORTS = PERSIST/'reports'
RAW       = Path('/content/data')
for d in (CROPS, MANIFESTS, CKPTS, REPORTS, RAW): d.mkdir(parents=True, exist_ok=True)
print("persisting to", PERSIST)

## 3 · Code

In [ ]:
import os
from pathlib import Path
REPO, BRANCH = Path('/content/Samsung_PRISM_Project_1'), 'claude/quirky-goodall-3ny0im'
if not REPO.exists():
    !git clone --depth 1 -b {BRANCH} https://github.com/RockRK2405/Samsung_PRISM_Project_1.git {REPO}
MODULE = REPO/'Samsung_PRISM_Video_Detection'
os.chdir(MODULE); print("cwd:", Path.cwd())
for f in ('scripts/prepare_from_registry.py','configs/dataset_registry.yaml'):
    print(('  OK      ' if (MODULE/f).exists() else '  MISSING '), f)

In [ ]:
# Run ONLY if the cell above printed MISSING. Upload the two files you were given.
import shutil
from google.colab import files
for name in files.upload():
    dest = (MODULE/'scripts'/name) if name.endswith('.py') else (MODULE/'configs'/name)
    dest.parent.mkdir(parents=True, exist_ok=True); shutil.move(name, dest)
    print("->", dest)

## 4 · Dependencies

`--no-deps` on facenet-pytorch is load-bearing — see fix 1 above. The
assertions fail loudly if anything downgrades numpy or breaks CUDA, because
both failures are otherwise silent until much later.

In [ ]:
!pip install -q --no-deps facenet-pytorch
!pip install -q timm opencv-python-headless pyyaml scikit-learn tqdm
!pip install -q -U kaggle          # 2.0.2 404s on DownloadDataset

import numpy, torch, torchvision, timm, cv2
print("numpy", numpy.__version__, "| torch", torch.__version__,
      "| torchvision", torchvision.__version__, "| timm", timm.__version__, "| cv2", cv2.__version__)
assert numpy.__version__.startswith("2"), "numpy was downgraded -> Runtime > Restart session, rerun"
assert torch.cuda.is_available(), "CUDA lost -> Runtime > Restart session, rerun"
!kaggle --version

In [ ]:
# MTCNN smoke test. Its weights load inside the constructor, so no exception = clear.
import numpy as np, torch
from facenet_pytorch import MTCNN
from PIL import Image
m = MTCNN(image_size=160, margin=20, device='cuda')
out = m(Image.fromarray((np.random.rand(480,640,3)*255).astype('uint8')))
print("MTCNN OK. noise ->", None if out is None else tuple(out.shape),
      "(None is EXPECTED: noise has no face)")

In [ ]:
# The project guards MTCNNFaceDetector behind a bare `except ImportError: pass`,
# so a missing dep would silently unexport it and fail deep inside prep instead.
import sys; sys.path.insert(0, '.')
from src.preprocessing import FrameExtractor, MTCNNFaceDetector
print("import guard OK -- both exported")

## 5 · Kaggle credentials

kaggle.com → Settings → API → **Create New Token**.

In [ ]:
import os, stat
from pathlib import Path
from google.colab import files
kdir = Path.home()/'.kaggle'; kdir.mkdir(exist_ok=True)
if not (kdir/'kaggle.json').exists():
    for n in files.upload(): os.replace(n, kdir/'kaggle.json')
(kdir/'kaggle.json').chmod(stat.S_IRUSR | stat.S_IWUSR)   # 600 or kaggle refuses
print("credentials in place")

## 6 · Celeb-DF v2 — the held-out test set (9.29 GB)

Downloaded first because it is the thing being measured. Expect **6529** mp4.

In [ ]:
import shutil
from pathlib import Path
RAW = Path('/content/data'); d = RAW/'celeb-df-v2'
have = sum(1 for _ in d.rglob('*.mp4')) if d.exists() else 0
if have > 6000:
    print(f"SKIP -- already have {have} mp4")
else:
    _,_,free = shutil.disk_usage('/content')
    assert free/1e9 > 25, f"only {free/1e9:.0f} GB free; need ~25 for zip+extract"
    d.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d reubensuju/celeb-df-v2 -p {d} --unzip
print("celeb-df-v2 mp4 files:", sum(1 for _ in d.rglob('*.mp4')), "(expect 6529)")
print("subdirs:", sorted(p.name for p in d.iterdir() if p.is_dir()))

## 7 · DFDC — training pool, **per part only**

The mirror is 96.5 GB total; `--unzip` needs the whole archive on disk first,
so the full download cannot fit. List the parts, then pull two (~20 GB).

In [ ]:
# Short listing (10 parts, not 300k frames) -- read the exact part names from this.
!kaggle datasets files pranay22077/dfdc-10 --page-size 50

In [ ]:
# Paste two part names from the listing above, then run.
PARTS = []          # e.g. ['dfdc_train_part_0.zip', 'dfdc_train_part_1.zip']

import shutil
from pathlib import Path
d = Path('/content/data/dfdc'); d.mkdir(parents=True, exist_ok=True)
assert PARTS, "Fill in PARTS from the listing above first."
for part in PARTS:
    _,_,free = shutil.disk_usage('/content')
    if free/1e9 < 25:
        print(f"STOP: only {free/1e9:.0f} GB free -- skipping {part}"); break
    print(f"\n=== {part}  ({free/1e9:.0f} GB free)")
    !kaggle datasets download -d pranay22077/dfdc-10 -f {part} -p {d} --unzip
print("\ndfdc mp4:", sum(1 for _ in d.rglob('*.mp4')),
      "| metadata.json:", sum(1 for _ in d.rglob('metadata.json')))

In [ ]:
# Disk + inventory check before the expensive step.
import shutil
from pathlib import Path
for p in sorted(x for x in Path('/content/data').iterdir() if x.is_dir()):
    n = sum(1 for _ in p.rglob('*.mp4'))
    gb = sum(f.stat().st_size for f in p.rglob('*') if f.is_file())/1e9
    print(f"  {p.name:16s} {n:6d} mp4  {gb:6.1f} GB")
_,_,free = shutil.disk_usage('/content'); print(f"\n{free/1e9:.0f} GB free")

## 8 · Prepare manifests — the expensive step (~30-60 min)

Samples frames, MTCNN-crops faces, writes `train.csv` / `val.csv` /
`test_celeb-df-v2.csv` + `splits_summary.json`.

Leakage rules **abort** rather than warn: no video in both train and val, no
`external_test` dataset in train. Frame-level splits are never produced.

Resumable via stable SHA-1 crop paths (fix 5), so a disconnect costs minutes.

In [ ]:
import time
t0 = time.time()
!python scripts/prepare_from_registry.py \
    --root dfdc=/content/data/dfdc \
    --root celeb-df-v2=/content/data/celeb-df-v2 \
    --out-dir "{CROPS}" --manifest-dir "{MANIFESTS}" \
    --videos-per-class 500 --frames-per-video 6 --val-fraction 0.15 \
    --device cuda --seed 42
print(f"\nprep: {(time.time()-t0)/60:.1f} min")

In [ ]:
import json
s = json.loads((MANIFESTS/'splits_summary.json').read_text())
print("train_pool   :", s['train_pool_datasets'])
print("external_test:", s['external_test_datasets'])
print("counts       :", s['counts'])
print("videos       :", s['train_videos'], "train /", s['val_videos'], "val")
print("leakage      :", s['leakage_checks'])
for k,v in s['per_dataset'].items(): print(f"  {k:16s} {v}")

assert s['counts']['train'] > 0, "no training crops -- check roots/globs"
assert s['external_test_datasets'], "no held-out test set -- val-only is meaningless"
n = s['counts']['train']
print(f"\n{n} train crops.", "OK." if n > 2000 else "THIN -- raise --videos-per-class.")

## 9 · Train (~40-80 min)

`--forensic-aug` applies JPEG recompression, resolution loss and mild blur —
the shift that collapsed EXP-005 from 0.99 in-distribution to 0.65
cross-dataset. It held FPR down there but did **not** recover recall, so treat
it as calibration help, not a cure. ConvNeXt-Tiny won EXP-005's backbone
comparison (0.6564 vs Xception 0.6407).

In [ ]:
import time
CKPT = CKPTS/'face_dfdc_convnext.pt'
t0 = time.time()
!python scripts/train_general.py \
    --train-manifest "{MANIFESTS}/train.csv" --val-manifest "{MANIFESTS}/val.csv" \
    --backbone convnext_tiny.fb_in22k_ft_in1k --forensic-aug \
    --epochs 8 --batch-size 64 --lr 3e-4 --image-size 224 \
    --num-workers 2 --device cuda --out "{CKPT}"
print(f"\ntrain: {(time.time()-t0)/60:.1f} min -> {CKPT}")

## 10 · Test on the held-out set

Threshold tuned on **val** for FPR ≤ 5% — tuning it on test would be leakage.

In [ ]:
import json
results = {}
for mf in sorted(MANIFESTS.glob('test_*.csv')):
    name = mf.stem.replace('test_','')
    out  = REPORTS/f'eval_{name}.json'
    print(f"\n{'='*64}\nTEST: {name}\n{'='*64}")
    !python scripts/eval_face_crops.py \
        --checkpoint "{CKPT}" --test-manifest "{mf}" \
        --val-manifest "{MANIFESTS}/val.csv" --target-fpr 0.05 \
        --batch-size 128 --num-workers 2 --out "{out}"
    if out.exists(): results[name] = json.loads(out.read_text())
print("\nscored:", list(results))

## 11 · Report

In [ ]:
import json

def dig(d, *keys):
    for k in keys:
        if isinstance(d, dict) and k in d: return d[k]
    for v in (d or {}).values():
        if isinstance(v, dict):
            got = dig(v, *keys)
            if got is not None: return got
    return None

fm = lambda v: f"{v:7.4f}" if isinstance(v,(int,float)) else f"{'n/a':>7s}"
print(f"{'dataset':18s} {'AUC':>7s} {'recall':>7s} {'fpr':>7s} {'f1':>7s} {'n':>7s}")
print('-'*60)
for f in sorted(REPORTS.glob('eval_*.json')):
    d = json.loads(f.read_text()); name = f.stem.replace('eval_','')
    print(f"{name:18s} {fm(dig(d,'auc','roc_auc'))} {fm(dig(d,'recall'))} "
          f"{fm(dig(d,'fpr'))} {fm(dig(d,'f1'))} {str(dig(d,'n') or '?'):>7s}")

print("\n".join(["",
 "Reading it honestly:",
 "  * Quote ROC-AUC, not F1. EXP-001 scored F1 0.81 while mis-flagging 72% of",
 "    real videos -- imbalance plus a fake-happy bias inflates F1 badly.",
 "  * 0.64-0.71 on an unseen dataset matches EXP-001/005. That is the",
 "    documented cross-dataset gap, not a bug in your run.",
 "  * 95% CI on AUC is ~+/-0.08 at n=100 crops, ~+/-0.03 at n=500. A 0.02",
 "    difference is not an improvement.",
 "  * You trained on DFDC, EXP-001 trained on FF++. This is a NEW data point,",
 "    not a replication of 0.7076. Say so in the write-up.",
]))

In [ ]:
# One run record, on Drive.
import json, time
out = REPORTS/f"run_{time.strftime('%Y%m%d_%H%M%S')}.json"
out.write_text(json.dumps({
    'run_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'protocol': 'train DFDC (parts) -> test held-out Celeb-DF v2',
    'splits': json.loads((MANIFESTS/'splits_summary.json').read_text()),
    'checkpoint': str(CKPT),
    'per_dataset': {f.stem: json.loads(f.read_text()) for f in REPORTS.glob('eval_*.json')},
}, indent=2, default=str))
print("wrote", out, "\nall artefacts under", PERSIST)

## 12 · Next

1. **Write it up as an EXP doc + metrics JSON**, positive or negative. Three of
   this project's five experiments are kept negatives; that is the convention.
2. **Add FF++ properly** via the official EULA at
   `github.com/ondyari/FaceForensics` — real mp4, the structure the registry
   expects, no Celeb-DF mixed in. That unlocks the direct 0.7076 comparison.
3. **More DFDC parts** for a bigger training pool: re-run cell 7 with new part
   names, then cell 8 (it resumes, so only new crops are extracted).
4. **DF40** is the one to chase. 40 manipulation methods supports
   leave-one-method-out — structurally the same as EXP-004's cross-generator
   design, the one setup in this project that actually generalized.
5. **Remember the ceiling.** No dataset here contains fully AI-generated video,
   so none of this moves general AI-video detection. That needs GenVidBench or
   a modern Sora/Veo/Kling set — separate work.